# Laboratorio 8 - SVM/SVR

## Desglose por ejercicio

Este notebook responde los ejercicios del laboratorio en formato **1 por 1**, siguiendo el estilo del avance.  
La idea es que primero se ejecute el script completo y luego se muestren, por seccion, las tablas, graficas y conclusiones correspondientes.


In [ ]:
from pathlib import Path
import json
import runpy
import pandas as pd
from IPython.display import display, Markdown, Image

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 200)

BASE_DIR = Path.cwd()
SCRIPT_PATH = BASE_DIR / 'lab8_laboratorio_resto.py'
OUTPUT_DIR = BASE_DIR / 'salidas'
FIG_DIR = OUTPUT_DIR / 'graficas'

runpy.run_path(str(SCRIPT_PATH), run_name='__main__')

with open(OUTPUT_DIR / 'train_test_split_metadata.json', 'r', encoding='utf-8') as f:
    split_meta = json.load(f)

classification_table = pd.read_csv(OUTPUT_DIR / 'classification_comparison_table.csv')
classification_overfit = pd.read_csv(OUTPUT_DIR / 'classification_overfitting_table.csv')
svm_table = pd.read_csv(OUTPUT_DIR / 'svm_candidate_table.csv')
regression_table = pd.read_csv(OUTPUT_DIR / 'regression_comparison_table.csv')
regression_overfit = pd.read_csv(OUTPUT_DIR / 'regression_overfitting_table.csv')
svr_table = pd.read_csv(OUTPUT_DIR / 'svr_candidate_table.csv')

best_svm_confusion = pd.read_csv(OUTPUT_DIR / 'confusion_matrices' / 'svm_best_test.csv', index_col=0)
best_rf_confusion = pd.read_csv(OUTPUT_DIR / 'confusion_matrices' / 'random_forest_best_test.csv', index_col=0)

best_class = classification_table.iloc[0]
recommended_class = classification_table[
    classification_table['Diagnóstico de ajuste'].isin(['Ajuste aceptable', 'Ligero sobreajuste'])
].sort_values(['F1_macro', 'Accuracy_test'], ascending=[False, False]).iloc[0]

best_reg = regression_table.iloc[0]
recommended_reg = regression_table[
    regression_table['Diagnóstico de ajuste'].isin(['Ajuste aceptable', 'Ligero sobreajuste'])
].sort_values(['RMSE_test', 'MAE_test', 'R2_test'], ascending=[True, True, False]).iloc[0]

selected_svm = classification_table[classification_table['Algoritmo'] == 'SVM'].iloc[0]
selected_svm_detail = svm_table[svm_table['modelo'] == selected_svm['Mejor modelo o configuración']].iloc[0]


## Ejercicio 1. Uso de los mismos conjuntos de entrenamiento y prueba

Se uso el mismo `train/test` del Laboratorio 8 para mantener una comparacion valida entre algoritmos.  
El split fue reproducible, estratificado por la variable categorica de precio y se reutilizo para todos los modelos.


In [ ]:
split_df = pd.DataFrame([
    ['Muestra total usada', split_meta['sample_size']],
    ['Entrenamiento', split_meta['train_size']],
    ['Prueba', split_meta['test_size']],
    ['Random state', split_meta['random_state']],
    ['Split reutilizado', 'Si'],
], columns=['Detalle', 'Valor'])

display(split_df)


## Ejercicio 2. Exploracion de datos y transformaciones necesarias

Para aplicar SVM y SVR se hicieron las transformaciones necesarias sobre el dataset:

- Limpieza de `price` para convertirlo a numerico.
- Imputacion de faltantes.
- Codificacion de variables categoricas con `OneHotEncoder`.
- Escalamiento de variables numericas con `StandardScaler`.
- Uso de `Pipeline` para evitar *data leakage*.

SVM necesita escalamiento porque trabaja con distancias y margenes; si una variable queda en una escala mucho mayor que otra, domina la separacion del hiperplano.


In [ ]:
transform_df = pd.DataFrame([
    ['Limpieza de precio', 'Se elimino simbolo de dolar y comas para crear price_num'],
    ['Imputacion numerica', 'Mediana'],
    ['Imputacion categorica', 'Valor mas frecuente'],
    ['Codificacion', 'One-hot encoding'],
    ['Escalamiento', 'StandardScaler en variables numericas'],
    ['Control de leakage', 'Imputadores y escaladores ajustados solo con train'],
], columns=['Transformacion', 'Descripcion'])

display(transform_df)


## Ejercicio 3. Variable respuesta categorica

La variable respuesta de clasificacion fue `price_cat`, con tres clases:

- `barata`
- `media`
- `cara`

Estas categorias se construyeron a partir de terciles del precio numerico.


In [ ]:
price_cat_df = pd.DataFrame([
    ['barata', 'Primer tercil del precio'],
    ['media', 'Segundo tercil del precio'],
    ['cara', 'Tercer tercil del precio'],
], columns=['Categoria', 'Criterio'])

display(price_cat_df)


## Ejercicio 4. Generacion de varios modelos SVM

Se evaluaron varias configuraciones de SVM con kernels:

- lineal
- RBF
- polinomial

Tambien se probaron distintos valores de `C`, `gamma` y `degree`. La seleccion final se hizo con validacion interna antes de revisar el test.


In [ ]:
display(svm_table[['modelo', 'kernel', 'hiperparametros', 'accuracy_test', 'f1_test', 'diagnostico']])


## Ejercicio 5. Prediccion de la variable respuesta

Cada modelo se entreno con el conjunto de entrenamiento y luego se predijo sobre `train` y `test`.  
Esto permitio comparar desempeno, estabilidad y posibles problemas de ajuste.


In [ ]:
display(pd.DataFrame([
    {
        'Modelo SVM seleccionado': selected_svm['Mejor modelo o configuración'],
        'Accuracy test': selected_svm['Accuracy_test'],
        'F1 macro test': selected_svm['F1_macro'],
        'Diagnostico': selected_svm['Diagnóstico de ajuste'],
    }
]))


## Ejercicio 6. Matrices de confusion

Se generaron matrices de confusion para los modelos de clasificacion y para todas las variantes SVM.  
En el SVM seleccionado, la clase mas dificil fue `media` y el error mas frecuente fue `cara -> media`.


In [ ]:
display(best_svm_confusion)
display(best_rf_confusion)

for fig_name, caption in [
    ('matriz_confusion_svm.png', 'Matriz de confusion del SVM seleccionado'),
    ('matriz_confusion_random_forest.png', 'Matriz de confusion del mejor modelo por test: Random Forest'),
]:
    display(Markdown(f'**{caption}**'))
    display(Image(filename=str(FIG_DIR / fig_name)))


## Ejercicio 7. Analisis de sobreajuste o desajuste

Para determinar si un modelo estaba sobreajustado o desajustado se compararon:

- `Accuracy train` vs `Accuracy test`
- `F1 train` vs `F1 test`
- diferencia entre train y test
- comportamiento por clase en la matriz de confusion

En clasificacion, el SVM recomendado presento **ligero sobreajuste**, mientras que KNN y Random Forest mostraron un sobreajuste mas marcado.


In [ ]:
display(classification_overfit[['Algoritmo', 'Configuración', 'Accuracy_train', 'Accuracy_test', 'F1_train', 'F1_test', 'Diferencia_F1', 'Diagnóstico']])
display(Image(filename=str(FIG_DIR / 'sobreajuste_clasificacion.png')))


## Ejercicio 8. Comparacion entre los modelos SVM

La comparacion entre variantes SVM se hizo considerando:

- efectividad (`Accuracy`, `F1 macro`)
- tiempo de entrenamiento y prediccion
- tipo de equivocaciones
- clase donde mas se equivoca y donde menos se equivoca

Las equivocaciones mas frecuentes se concentraron entre `media` y `cara`, lo cual es menos grave que confundir directamente una casa `cara` como `barata`.


In [ ]:
display(svm_table[['modelo', 'accuracy_test', 'f1_test', 'tiempo_entrenamiento_seg', 'tiempo_prediccion_seg', 'confusion_mas_frecuente', 'mejor_clase_test', 'peor_clase_test', 'diagnostico']])
display(Image(filename=str(FIG_DIR / 'svm_f1_variantes.png')))


## Ejercicio 9. Comparacion del mejor SVM con modelos anteriores de clasificacion

Se comparo el mejor SVM con:

- Arbol de decision
- Random Forest
- Naive Bayes
- KNN
- Regresion logistica

Todos los modelos fueron reentrenados con la misma variable respuesta y el mismo split.


In [ ]:
display(classification_table)
display(Image(filename=str(FIG_DIR / 'clasificacion_accuracy_f1.png')))

print('Mejor por test:', best_class['Algoritmo'], '-', best_class['Mejor modelo o configuración'])
print('Recomendado por equilibrio:', recommended_class['Algoritmo'], '-', recommended_class['Mejor modelo o configuración'])


## Ejercicio 10. Tabla comparativa de sobreajuste en clasificacion

Los parametros clave para detectar sobreajuste en clasificacion fueron:

- `Accuracy train` frente a `Accuracy test`
- `F1 train` frente a `F1 test`
- brecha entre ambas metricas
- errores por clase en la matriz de confusion

KNN y Random Forest tendieron a sobreajustarse mas que el SVM recomendado.


In [ ]:
display(classification_overfit)


## Ejercicio 11. Generacion de un modelo de regresion con el precio directo

Para regresion se uso directamente la variable `price_num`.  
Se evaluaron modelos SVR con kernel lineal, RBF y polinomial, tuneando `C`, `gamma`, `epsilon` y `degree` cuando aplicaba.


In [ ]:
display(svr_table)
display(Image(filename=str(FIG_DIR / 'svr_rmse_variantes.png')))


## Ejercicio 12. Comparacion del modelo de regresion con modelos anteriores

El modelo SVR se comparo con:

- Regresion lineal
- Arbol de regresion
- Random Forest Regressor
- KNN Regressor
- Naive Bayes aproximado para regresion

La comparacion se hizo principalmente con `MAE`, `RMSE`, `R2` y estabilidad entre entrenamiento y prueba.


In [ ]:
display(regression_table)
display(regression_overfit)
display(Image(filename=str(FIG_DIR / 'regresion_rmse_test.png')))
display(Image(filename=str(FIG_DIR / 'sobreajuste_regresion.png')))

print('Mejor por test:', best_reg['Algoritmo'], '-', best_reg['Mejor configuración'])
print('Recomendado por equilibrio:', recommended_reg['Algoritmo'], '-', recommended_reg['Mejor configuración'])


## Conclusiones

- Mejor clasificacion por metrica de prueba: `Random Forest`.
- Clasificacion recomendada por equilibrio: `SVM`.
- Mejor regresion por metrica de prueba: `KNN Regressor`.
- Regresion recomendada por equilibrio: `Random Forest Regressor`.

Este notebook ya queda con el mismo estilo del avance: secciones separadas por ejercicio, explicacion breve y evidencia mostrada directamente en celdas.
